# Reproducible results

Every result can be reproduced exactly: jobs are seeded, the server echoes the seed it drew when none was given, and
each noisy response lists the calibration values it used and which of them are assumed rather than published.

In [1]:
from qputwin_demo import *

connect()

QPUTwin at http://localhost:8000: ready, 7 twins reachable


{'status': 'ready',
 'instance': 'c88add07efb8:7:9e791f',
 'job_store': 'ok',
 'backends_reachable': ['quantinuum_helios',
  'ionq_forte',
  'rigetti_ankaa_3',
  'iqm_emerald',
  'ibm_heron',
  'ibm_nighthawk',
  'google_willow']}

## 1. Same seed, same counts

With a fixed seed, ideal and noisy jobs return identical counts, on every run.

In [2]:
for noise in (None, True):
    a = run("iqm_emerald", ghz(5), shots=5000, seed=123, noise=noise)["result"]["counts"]
    b = run("iqm_emerald", ghz(5), shots=5000, seed=123, noise=noise)["result"]["counts"]
    print("noisy" if noise else "ideal", "identical:", a == b)

ideal identical: True
noisy identical: True


## 2. Replaying a server-drawn seed

Without a seed the server draws one and returns it; sending it back reproduces the result.

In [3]:
first = run("ionq_forte", ghz(4), shots=2000, seed=None, noise=True)
seed = first["result"]["seed"]
replay = run("ionq_forte", ghz(4), shots=2000, seed=seed, noise=True)
print("server-drawn seed:", seed)
print("replay identical:", first["result"]["counts"] == replay["result"]["counts"])

server-drawn seed: 5270841020382865606
replay identical: True


## 3. Every shot, in order

`memory: true` returns each shot's bitstring in order, for analyses that need more than counts.

In [4]:
r = run("quantinuum_helios", bell(), shots=20, memory=True)
print("".join(s[0] for s in r["result"]["memory"]), "<- first bit of each shot")
print("first bit equals second bit in every shot:", all(s[0] == s[1] for s in r["result"]["memory"]))

10100110101011110011 <- first bit of each shot
first bit equals second bit in every shot: True


## 4. What the result depended on

The noise report lists every calibration value used, and flags those that are assumed rather than published; report them together with the result.

In [5]:
r = run("ionq_forte", ghz(3), shots=1000, noise=True)
n = r["result"]["noise"]
print("calibration:", n["calibration"], "| origin:", n["calibration_origin"])
print("assumed parameters:", n["assumed_parameters"])
for tag, paths in n["parameters_used"].items():
    print(f"{tag:10s} {len(paths):2d}: {', '.join(paths)}")

calibration: IonQ Forte - nominal medians | origin: packaged
assumed parameters: ['coherence.t1_us', 'coherence.t2_us', 'gates.gpi2.duration_ns', 'gates.ms.duration_ns']
published   3: gates.gpi2.fidelity, gates.ms.fidelity, readout.fidelity
assumed     4: coherence.t1_us, coherence.t2_us, gates.gpi2.duration_ns, gates.ms.duration_ns
untagged    0: 


## 5. The software environment

The telemetry records the versions that produced the result, for the methods section of a paper.

In [6]:
env = r["telemetry"]["environment"]
{k: env[k] for k in env if k != "worker"} | {"worker": {k: v for k, v in env["worker"].items() if k in ("vendor", "numpy", "python", "blas_threads", "cgroup_cpu_limit", "coupling_map")}}

{'orchestrator_version': '0.1.0',
 'python': '3.13.16',
 'replica': 'http://172.20.0.2:8001',
 'worker': {'vendor': 'ionq_forte',
  'python': '3.13.16',
  'numpy': '2.5.4',
  'blas_threads': '1',
  'cgroup_cpu_limit': 4.0,
  'coupling_map': 'packaged'}}

## Takeaways

- A seed fully determines a result, ideal or noisy.
- Responses carry the calibration provenance and software versions needed to report a result.